# D1-S2-Nb3: The Powers of Low-Level Probing
Showcase:
- state-verification
- auto-correction / self-healing aPs

## Set your `protocol_mode`

...so it is easy to switch between simulation and execution

In [1]:
prep_simulation = False  # True or False

star_simulation = True  # True or False

run_identifier = "nb3"

## Library Import

In [2]:
import asyncio
import logging
import time

---
## Visualizer3D
...so everything that follows has one highest-level reference frame

In [3]:
from pylabrobot.resources import Coordinate, Resource
from pylabrobot.visualizer3D.facility import Facility
from pylabrobot.visualizer3D.server import Viewer3D

facility = Facility(name="facility", size_x=2_400, size_y=1_000, size_z=0)

viewer = Viewer3D(root = facility, name="nb3_probing")

await viewer.start()

viewer on http://127.0.0.1:1339/#token=4WIxXBPn2aIXx28vrLTKAp0TKhLxSVi1CvrqPGq1uU0  (websocket 2123)


viewer: a browser connected, drawing with WebGPU


---

## Set your `logging`
We (and our agents) always benefit from a log.

This follows standard Python logging but for convenience we set up:
- showing logging inside notebook using `verbose` declaration
- simple way to log to a file (in append mode)

I recommend always logging all commands you send/receive from any device for auditing purposes.

In [4]:
import datetime
import logging
import time

import pylabrobot

from pylabrobot.io import LOG_LEVEL_IO

# one mode per device: 4 combinations, each with its own log folder
prep_mode = "simulation" if prep_simulation else "execution"
star_mode = "simulation" if star_simulation else "execution"
protocol_mode = f"prep_{prep_mode}-star_{star_mode}"

started_at = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

log_path = f"_logs/{protocol_mode}/{run_identifier}_{started_at}.log"
pylabrobot.setup_logger(log_path, level=LOG_LEVEL_IO)

pylabrobot.verbose(True, level=logging.INFO)

print(f"logging to {log_path}")
logging.getLogger("pylabrobot").info("--- %s (%s) ---", run_identifier, protocol_mode)

2026-09-28 11:08:48,705 - pylabrobot - INFO - --- nb3 (prep_execution-star_simulation) ---


logging to _logs/prep_execution-star_simulation/nb3_20260928-110848.log


## Device setups
### (shadow) STAR(let)

In [5]:
from pylabrobot.hamilton.star import STARlet

star = STARlet(simulation=star_simulation)

facility.assign_child_resource(
    resource=star,
    location=Coordinate(x=800.0, y=0.0, z=0.0)
)

await star.setup()

2026-09-28 11:09:06,671 - pylabrobot.hamilton.star.driver.master - INFO - [Hamilton STAR] Connected on simulation (no link)
  Firmware: master 7.6S 25 2021_11_05 (GRU C0), pipettes 4.0S j 2022-03-16, x_arm 1.4S 2012-04-25, head96 5.0S i 2021-10-22 (H0 XE167), iswap 4.1S 2011-12-19, autoload 3.4S f 2017-01-09
  Arms: 1
  Configuration: 30 slots
  Autoload: 1D barcode scanner
    left: hamilton_legacy_star_dual_rail_arm, 354.0 mm wide, travel 95.0 to 800.2 mm, workspace -323.2 to 977.2 mm
      channels: 8 (1000uL) | 96-head: 96 head II | 384-head: none | iSWAP: wide gripper


### Prep

In [6]:
from pylabrobot.hamilton.prep import Prep

HOST = "192.168.100.102"  # ignored in simulation
PORT = 2000

prep = Prep(
    simulation=prep_simulation,
    host=HOST,
    port=PORT,
    )

facility.assign_child_resource(
    prep,
    location=Coordinate(x=0.0, y=0.0, z=0.0)
)

await prep.setup()
await prep.lights.turn_on()

2026-09-28 11:10:33,029 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Initializing Hamilton connection...
2026-09-28 11:10:33,315 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Registering Hamilton client...
2026-09-28 11:10:33,316 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Discovering Hamilton root objects...
2026-09-28 11:10:33,317 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Discovering Hamilton global objects...
2026-09-28 11:10:33,320 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Hamilton TCP client setup complete. Client ID: 5, globals: 1
2026-09-28 11:10:33,386 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Initializing Hamilton connection...
2026-09-28 11:10:33,886 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Registering Hamilton client...
2026-09-28 11:10:33,889 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Discovering Hamilton root objects...
2026-09-28 11:10:33,891 - pylabrobot.hamilton.transport.tcp.tcp - INFO - Discovering Hamilton global objects

2026-09-28 11:20:34,359 - pylabrobot.hamilton.transport.tcp.session - WARNING - Dropping unmatched response from 1:1:48900 to 2:5:65535 seq=0 action=0x7
2026-09-28 11:20:34,360 - pylabrobot.hamilton.transport.tcp.session - WARNING - Dropping unmatched response from 1:1:48900 to 2:6:65535 seq=0 action=0x7
2026-09-28 11:20:34,361 - pylabrobot.hamilton.transport.tcp.session - WARNING - Hamilton TCP reader stopped: Connection closed before num_bytes are read
2026-09-28 11:44:26,472 - pylabrobot.hamilton.transport.tcp.session - WARNING - Dropping unmatched response from 1:1:48900 to 2:5:65535 seq=0 action=0x7
2026-09-28 11:44:26,474 - pylabrobot.hamilton.transport.tcp.session - WARNING - Hamilton TCP reader stopped: Connection closed before num_bytes are read


---
## Pickup probing tools
...all Hamilton liquid handling workstations come with "teaching needles" / metal tips attached to some part of the deck (usually the waste block)

you can easily find them by using PLR's `get_resource()` finding function:

In [7]:
prep_tipspot = prep.get_resource("Prep_teaching_needle")
prep_tipspot

TipSpot(name='Prep_teaching_needle', location=Coordinate(002.510, 218.540, 075.750), size_x=6.0, size_y=6.0, size_z=0.0, category=teaching_needle)

On the STAR there is not just *one* teaching needle, but an entire tiprack of teaching needles:

In [8]:
star_teaching_tiprack = star.get_resource("STARlet_deck_teaching_tip_rack")
star_teaching_tiprack

TipRack(name='STARlet_deck_teaching_tip_rack', size_x=9, size_y=72, size_z=50.4, location=Coordinate(005.900, 346.100, 000.000))

Having the tipspots/tiprack readily available we can pick them up:

In [9]:
prep_channel_idx = 1 # easier to see for you guys in the audience ;)

await prep.pipettes.pick_up_tips(
    tip_spots=[prep_tipspot], # has to be a list
    use_channels=[prep_channel_idx]
)

In [10]:
prep.get_resource("Prep_calibration_block").get_location_wrt(prep.deck,"c","c","t")

Coordinate(x=132.55, y=186.73, z=22.0)

In [11]:
await prep.pipettes.probe_calibration_block_cct(channel_idx=1)

Coordinate(x=133.7, y=187.73, z=20.34)

In [12]:
Coordinate(x=133.7, y=187.73, z=20.34) - Coordinate(x=132.55, y=186.73, z=22.0)


Coordinate(x=1.15, y=1.0, z=-1.66)

In [13]:
prep.update_deck_components(offset=Coordinate(x=1.15, y=1.0, z=-1.66))

Since, even on the Prep, there are multiple pipettes - and we *could* therefore pick up multiple needles/tips - arguments always take a list.

The index of an element in the list declares which pipette it addresses.

N.B.: Python uses 0-based indexing! And therefore so does PyLabRobot.

N.B.2: Physical pipettes are counted from back to front, starting at 0!

In [14]:
star_channel_idxs = [6, 7] # easier to see for you guys in the audience ;)

await star.pipettes.pick_up_tips(
    tip_spots=star_teaching_tiprack["A1:B1"],
    use_channels=star_channel_idxs
)

In [18]:
[x+1 for x in range(5)]

[1, 2, 3, 4, 5]

---
## Probing

### ... the Deck

move pipettes to the position you want to probe, then call ztouch probing method:

#### on Prep
The Prep has a handy little calibration block at a known position for calibration purposes:


In [19]:
calibration_block = prep.get_resource("Prep_calibration_block")
calibration_block

Resource(name='Prep_calibration_block', location=Coordinate(129.550, 183.730, 000.000), size_x=6.0, size_y=6.0, size_z=22.0, category=calibration_block)

To `get_` its center_x, center_y, top position we can use PLR's resource model:

In [20]:
calibration_block_cct = calibration_block.get_location_wrt(
        other=prep.deck, # we always move motors relative to the deck
        x="c",
        y="c",
        z="t"
    )

calibration_block_cct

Coordinate(x=132.55, y=186.73, z=22.0)

In [25]:
await prep.pipettes.move_to_xy_positions(
    x = calibration_block_cct.x,
    ys = {prep_channel_idx : calibration_block_cct.y}
)

In [28]:
measure_results = []

start_z = 40

for n in range(5):

    measured_z = await prep.pipettes.probe_z_using_ztouch(
        channel_idx=prep_channel_idx,
        search_start_position = start_z,
        # search_end_position: Optional[float] = None,
        # search_speed: float = 10.0,
        # push_force_pwm: Optional[int] = None,
        # end_tolerance: float = 1.2,
        # tip_len: Optional[float] = None,
        # allow_without_tip: bool = False,
        # post_detection_distance: float = 2.0,
        # move_to_safe_z_position_after: bool = False,
    )

    start_z = measured_z + 5

    measure_results.append(measured_z)
    print(n, measured_z)

print(f"Prep: {measured_z=}")

await prep.pipettes.move_to_safe_z()

measure_results

0 20.41
1 20.37
2 20.38
3 20.4
4 20.37
Prep: measured_z=20.37


[20.41, 20.37, 20.38, 20.4, 20.37]

#### (Almost) the same for STAR:

In [29]:
chosen_deck_coordinates = [
    Coordinate(400.0, 500.0, 245.0),
    Coordinate(400.0, 200.0, 245.0),
]
chosen_ys = [coord.y for coord in chosen_deck_coordinates]


await star.pipettes.move_to_xy_positions(
    x = chosen_deck_coordinates[0].x,
    ys = dict(zip(star_channel_idxs, chosen_ys))
)


measured_zs = await asyncio.gather(
    *(
        star.pipettes.probe_z_using_ztouch(
            channel=ch
        )
        for ch in star_channel_idxs)
  )
print(f"STARlet: {measured_zs=}")

await star.pipettes.move_to_safe_z()

STARlet: measured_zs=[None, None]


### ... in multiple locations 

In [30]:
await prep.pipettes.probe_deck_corner_z_offsets()

{'30.0,-2.5': -1.4,
 '30.0,375.0': 28.44,
 '235.0,-2.5': -0.87,
 '235.0,375.0': -1.56}

then we want to ensure we place the teaching needle back:

In [31]:
# via an explicit drop command:

# await prep.pipettes.drop_tips(
#     tip_spots=[prep_tipspot], # has to be a list
#     use_channels=[prep_channel_idx]
# )

# or the lazy way :P

await prep.pipettes.return_tips()

The PLR STAR(let) does not yet have this deck probing convience method.

...but we still have to return the teaching needles on the STARlets:

In [32]:
await star.pipettes.return_tips()

---
### ... tip presence

using the sleeve sensors of the pipettes we can check whether a tip exists at a tipspot.

i.e. we can verify the state of the resource model
...and correct it if the model is outdated!

But first, we need some tips and activate `set_tip_tracking`:

In [33]:
from pylabrobot.resources import set_tip_tracking
from pylabrobot.resources.hamilton import (
    hamilton_96_tiprack_50uL_NTR,
    hamilton_96_tiprack_300uL_NTR
)

set_tip_tracking(enabled=True)

then create new tipracks and assign them to the deck:

In [34]:
ntr_50ul_00 = prep.deck[1] = hamilton_96_tiprack_50uL_NTR(
    name="ntr_50ul_00",
    )

ntr_300ul_00 = prep.deck[7] = hamilton_96_tiprack_300uL_NTR(
    name="ntr_300ul_00",
    )

In [35]:
ntr_50ul_00

StandingTipRack(name='ntr_50ul_00', size_x=127.76, size_y=85.48, size_z=55.0, stacking_z_height=16.0, location=Coordinate(002.870, 003.010, 003.500))

In [ ]:
start_time = time.time()

tip_probing_results = await prep.pipettes.probe_tip_presence_via_pickup(
    tip_spots=ntr_50ul_00["G4:H4"]
)

end_time = time.time() - start_time

tip_probing_results, f"duraction: {end_time} sec"

AttributeError: 'Pipettes' object has no attribute 'probe_tip_presence_via_pickup'

#### Update if model is outdated

In [ ]:
tip_probing_results = await prep.pipettes.probe_tip_presence_via_pickup(
    tip_spots=ntr_50ul_00["G1:H1"]
)

for spot_name, has_tip in tip_probing_results.items():
    print(spot_name, has_tip)

    if not has_tip:
        # the model said there was a tip,
        # the device found none:
        print(f" -> correct the model's tipstate")
        ntr_50ul_00.get_resource(spot_name).unassign_tip()

same on the STAR:

In [40]:
from pylabrobot.lib.liquid_handling.tip_presence_probing import probe_tip_presence_via_pickup
from pylabrobot.resources.hamilton import hamilton_tip_carrier_L5_ntr_a00


tip_carrier = hamilton_tip_carrier_L5_ntr_a00(name="star_tip_carrier")
tip_carrier[0] = star_ntr_50ul_00 = hamilton_96_tiprack_50uL_NTR(name="star_ntr_50ul_00")
star.deck.assign_child_resource(tip_carrier, track=13)

ValueError: Resource with name 'star_tip_carrier' already defined.

In [37]:
# the STAR has no `probe_tip_presence_via_pickup` of its own yet: the shared helper drives its calls
tip_probing_results = await probe_tip_presence_via_pickup(
    tip_spots=star_ntr_50ul_00["G1:H1"],
    use_channels=star_channel_idxs,
    pick_up_tips=lambda spots, channels: star.pipettes.pick_up_tips(spots, use_channels=channels),
    drop_tips=lambda spots, channels: star.pipettes.drop_tips(spots, use_channels=channels),
    sense_tip_presence=star.pipettes.sense_tip_presence,
)

for spot_name, has_tip in tip_probing_results.items():
    print(spot_name, has_tip)

    if not has_tip:
        print(f" -> correct the model's tipstate")
        star_ntr_50ul_00.get_resource(spot_name).unassign_tip()

NameError: name 'probe_tip_presence_via_pickup' is not defined

## STAR only: carrier sensing

In [41]:
await star.autoload.sense_carrier_presence_on_deck()

[18]

In [42]:
await star.autoload.sense_carrier_presence_on_loading_tray()
# N.B.: this will move the autoload back and forth
# along the entire x-length of the device

[]

In [43]:
for track_idx in [40, 39, 41]:
    sense_result = await star.autoload.sense_carrier_presence_on_single_loading_tray_track(
        track=track_idx,
        park_after=False
    )
    print(f"{track_idx}: track present? -> {sense_result}")

await star.autoload.park()

40: track present? -> False
39: track present? -> False
41: track present? -> False


---
# Shutdown

In [44]:
await star.stop()

In [45]:
await prep.stop()

2026-09-28 12:01:52,390 - pylabrobot.hamilton.prep.driver.master - ERROR - could not read whether a plate is held; treating it as held
Traceback (most recent call last):
  File "/Users/camillomoschner/Documents/GitHub/pylabrobot/pylabrobot/hamilton/prep/driver/master.py", line 1215, in _return_held_resource
    if not await self._request_plate_held():
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/camillomoschner/Documents/GitHub/pylabrobot/pylabrobot/hamilton/prep/driver/master.py", line 1245, in _request_plate_held
    return bool((await self.send_command(PrepCmd.PrepGetPlateHeld())).value)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/camillomoschner/Documents/GitHub/pylabrobot/pylabrobot/hamilton/prep/driver/master.py", line 789, in send_command
    resolved = await self._resolve_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/camillomoschner/Documents/GitHub/pylabrobot/pylabrobot/hamilton/prep/dr

In [46]:
await viewer.stop()